# 01 EDA

Exploratory data analysis for the breast cancer clinical decision support system.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.feature_engineering import class_imbalance_report, correlation_analysis, detect_outliers_iqr
from src.utils import DEFAULT_DATA_PATH, get_feature_columns, load_dataset, save_json, validate_dataset

sns.set_theme(style='whitegrid')
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

In [ ]:
df = load_dataset(DEFAULT_DATA_PATH)
df.head()

In [ ]:
validation_report = validate_dataset(df)
save_json(validation_report, REPORTS_DIR / 'eda_validation_report.json')
validation_report

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()
print('Duplicate rows:', duplicates)
missing[missing > 0]

In [ ]:
imbalance = class_imbalance_report(df)
imbalance

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='diagnosis', palette={'B': '#0f766e', 'M': '#b42318'}, hue='diagnosis', legend=False)
plt.title('Class Distribution')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'notebook_class_distribution.png', dpi=180)
plt.show()

In [ ]:
feature_columns = get_feature_columns(df)
df[feature_columns].describe().T

In [ ]:
selected_features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'concavity_mean', 'concave points_mean']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.ravel(), selected_features):
    sns.histplot(data=df, x=feature, hue='diagnosis', kde=True, ax=ax, palette={'B': '#0f766e', 'M': '#b42318'})
    ax.set_title(feature)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'notebook_feature_histograms.png', dpi=180)
plt.show()

In [ ]:
corr, high_corr = correlation_analysis(df)
plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': 0.75})
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'notebook_correlation_heatmap.png', dpi=180)
plt.show()
high_corr.head(20)

In [ ]:
pairplot_features = ['diagnosis', 'radius_mean', 'texture_mean', 'area_mean', 'smoothness_mean', 'concavity_mean']
plot = sns.pairplot(df[pairplot_features], hue='diagnosis', palette={'B': '#0f766e', 'M': '#b42318'}, corner=True)
plot.fig.suptitle('Selected Feature Pairplot', y=1.02)
plot.savefig(REPORTS_DIR / 'notebook_pairplot.png', dpi=160)

In [ ]:
outliers = detect_outliers_iqr(df)
outliers.to_csv(REPORTS_DIR / 'notebook_outlier_report.csv', index=False)
outliers.head(15)